<a href="https://colab.research.google.com/github/Ashokyadhav/Econocausal/blob/ashok/DoubleML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
df = pd.read_csv("/content/wcer (1).csv")

In [5]:


print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())


Dataset shape:
(4751202, 5)

Column names:
['STORE', 'UPC', 'WEEK', 'MOVE', 'PRICE']

First 5 rows:


,STORE,UPC,WEEK,MOVE,PRICE
0,51,317,372,1,26.020
1,74,317,367,1,19.060
2,93,317,364,2,1.475
3,104,317,383,1,3.170
4,111,317,381,1,10.460



Missing values:
STORE    0
UPC      0
WEEK     0
MOVE     0
PRICE    0
dtype: int64


In [6]:

# Step 3: Understand our dataset

print("Dataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe())

print("\nNumber of unique stores:", df["STORE"].nunique())
print("Number of unique products:", df["UPC"].nunique())
print("Number of unique weeks:", df["WEEK"].nunique())


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4751202 entries, 0 to 4751201
Data columns (total 5 columns):
 #   Column  Dtype  
---  ------  -----  
 0   STORE   int64  
 1   UPC     int64  
 2   WEEK    int64  
 3   MOVE    int64  
 4   PRICE   float64
dtypes: float64(1), int64(4)
memory usage: 181.2 MB

Statistical summary:


,STORE,UPC,WEEK,MOVE,PRICE
count,4.751202e+06,4.751202e+06,4.751202e+06,4.751202e+06,4.751202e+06
mean,8.345552e+01,3.756107e+09,1.960909e+02,1.956532e+01,3.114450e+00
std,3.651187e+01,7.449993e+09,1.149442e+02,5.872462e+01,7.643148e-01
min,2.000000e+00,3.170000e+02,1.000000e+00,1.000000e+00,5.000000e-02
25%,5.600000e+01,1.600069e+09,1.000000e+02,7.000000e+00,2.650000e+00
50%,8.900000e+01,3.800001e+09,1.860000e+02,1.300000e+01,3.140000e+00
75%,1.130000e+02,3.800013e+09,3.100000e+02,2.100000e+01,3.590000e+00
max,1.460000e+02,7.916692e+10,3.990000e+02,1.868800e+04,2.602000e+01



Number of unique stores: 93
Number of unique products: 489
Number of unique weeks: 366


In [7]:

# Step 5: Check MOVE distribution

print("MOVE percentiles:")
display(df["MOVE"].quantile([0.50, 0.90, 0.95, 0.99, 0.999]))

print("\nLargest 10 MOVE values:")
display(df.nlargest(10, "MOVE")[["STORE", "UPC", "WEEK", "MOVE", "PRICE"]])


MOVE percentiles:


,MOVE
0.500,13.0
0.900,34.0
0.950,46.0
0.990,135.0
0.999,569.0



Largest 10 MOVE values:


,STORE,UPC,WEEK,MOVE,PRICE
767425,44,1600066610,13,18688,2.68
2265798,44,3800000510,13,17920,1.85
1922880,86,3000006560,395,17824,1.50
2154815,44,3800000110,13,15360,1.49
2727780,44,3800001611,13,15104,2.39
827749,44,1600066750,13,14080,3.57
657555,44,1600066310,13,12800,1.85
2293603,44,3800000520,13,12544,2.20
2214827,44,3800000127,13,12288,2.41
741901,64,1600066590,30,11776,2.85


In [8]:

# Step 6: Check the relationship between PRICE and MOVE

print("PRICE summary:")
print(df["PRICE"].describe())

print("\nCorrelation between PRICE and MOVE:")
print(df[["PRICE", "MOVE"]].corr())


PRICE summary:
count    4.751202e+06
mean     3.114450e+00
std      7.643148e-01
min      5.000000e-02
25%      2.650000e+00
50%      3.140000e+00
75%      3.590000e+00
max      2.602000e+01
Name: PRICE, dtype: float64

Correlation between PRICE and MOVE:
          PRICE      MOVE
PRICE  1.000000 -0.099806
MOVE  -0.099806  1.000000


In [9]:

# Step 8: Average price by week

weekly_price = (
    df.groupby("WEEK")["PRICE"]
    .mean()
    .sort_index()
)

print("First 10 weeks:")
display(weekly_price.head(10))

print("\nLast 10 weeks:")
display(weekly_price.tail(10))

print("\nNumber of weeks with price data:", weekly_price.count())


First 10 weeks:


,PRICE
WEEK,
1,2.605674
2,2.605345
3,2.605234
4,2.621838
5,2.623008
6,2.621803
7,2.630338
8,2.629094
9,2.636263



Last 10 weeks:


,PRICE
WEEK,
390,3.194828
391,3.190778
392,3.188336
393,3.160984
394,3.160984
395,3.152996
396,3.182525
397,3.157225
398,3.174431



Number of weeks with price data: 366


In [10]:

# Step 9: Check price variation within products

product_price = (
    df.groupby("UPC")["PRICE"]
    .agg(["min", "mean", "max", "std"])
)

print("Products with the highest price variation:")
display(
    product_price.sort_values("std", ascending=False).head(10)
)

print("\nProducts with no price variation:")
print((product_price["std"].fillna(0) == 0).sum())


Products with the highest price variation:


,min,mean,max,std
UPC,,,,
317,1.475,12.037000,26.02,10.458409
3800054268,3.720,6.229333,6.49,0.699945
79166919305,1.660,3.093603,3.58,0.693711
1780019541,1.680,2.929835,3.49,0.688500
3800001727,0.090,4.353422,5.05,0.685618
3828125030,1.490,2.626796,3.39,0.682712
1780010605,0.740,3.567400,3.99,0.572462
3800013930,1.340,4.367080,5.40,0.571532
79166910359,0.640,3.662361,3.99,0.560446



Products with no price variation:
42


In [11]:

# Step 10: Check price variation within each store-product pair

store_product_price = (
    df.groupby(["STORE", "UPC"])["PRICE"]
    .agg(["min", "max", "std", "count"])
)

store_product_price["price_range"] = (
    store_product_price["max"] - store_product_price["min"]
)

print("Total store-product pairs:", len(store_product_price))

print("\nPairs with no price variation:")
print((store_product_price["price_range"] == 0).sum())

print("\nPairs with price variation:")
print((store_product_price["price_range"] > 0).sum())

print("\nExamples with the highest price variation:")
display(
    store_product_price.sort_values(
        "price_range", ascending=False
    ).head(10)
)


Total store-product pairs: 36494

Pairs with no price variation:
5292

Pairs with price variation:
31202

Examples with the highest price variation:


min   max       std  count  price_range
STORE UPC                                                 
100   3800004900  1.64  9.71  0.550870    361         8.07
98    3800001727  0.09  4.89  0.675770    162         4.80
      3800002970  0.10  4.79  0.617819    105         4.69
59    3800000127  0.69  4.96  0.464022    362         4.27
128   3800054276  3.29  7.38  0.993243     20         4.09
114   3800054276  2.93  6.98  0.593332     49         4.05
112   3800001510  0.99  5.00  0.371437    366         4.01
115   3800001510  0.99  5.00  0.392843    364         4.01
51    3800001510  0.99  5.00  0.399313    359         4.01
95    3828125069  0.99  5.00  0.513058    219         4.01

In [12]:

# Step 11: Check duplicate store-product-week combinations

duplicate_count = df.duplicated(
    subset=["STORE", "UPC", "WEEK"]
).sum()

print("Duplicate store-product-week rows:", duplicate_count)

print("\nExample duplicate records:")
display(
    df[
        df.duplicated(
            subset=["STORE", "UPC", "WEEK"],
            keep=False
        )
    ].head(10)
)


Duplicate store-product-week rows: 0

Example duplicate records:


,STORE,UPC,WEEK,MOVE,PRICE


In [13]:

# Step 13: Check records per store and product

store_counts = df["STORE"].value_counts()
product_counts = df["UPC"].value_counts()

print("Records per store:")
print(store_counts.describe())

print("\nRecords per product:")
print(product_counts.describe())

print("\nStores with the most records:")
display(store_counts.head(10))

print("\nProducts with the most records:")
display(product_counts.head(10))


Records per store:
count       93.000000
mean     51088.193548
std      14410.693032
min       4521.000000
25%      53059.000000
50%      56580.000000
75%      58350.000000
max      62575.000000
Name: count, dtype: float64

Records per product:
count      489.000000
mean      9716.159509
std      10534.490437
min          1.000000
25%       1093.000000
50%       4771.000000
75%      17080.000000
max      30100.000000
Name: count, dtype: float64

Stores with the most records:


,count
STORE,
126,62575
74,61560
121,61364
132,61323
131,60940
110,60844
98,60843
59,60582
102,60188



Products with the most records:


,count
UPC,
3800001912,30100
3800000120,30077
3800001621,30074
3800001611,30073
1600065700,30036
3800001411,30033
1600065890,30029
1600066210,30027
3800001720,30025


In [14]:

# Step 14: Check observations per store-product pair

pair_counts = df.groupby(["STORE", "UPC"]).size()

print("Store-product pair summary:")
print(pair_counts.describe())

print("\nPairs with fewer than 50 observations:")
print((pair_counts < 50).sum())

print("\nPairs with at least 300 observations:")
print((pair_counts >= 300).sum())

print("\nPairs with the fewest observations:")
display(pair_counts.sort_values().head(10))


Store-product pair summary:
count    36494.000000
mean       130.191319
std        124.304612
min          1.000000
25%         26.000000
50%         78.000000
75%        227.000000
max        366.000000
dtype: float64

Pairs with fewer than 50 observations:
14588

Pairs with at least 300 observations:
6284

Pairs with the fewest observations:


,,0
STORE,UPC,
84,3800032100,1
123,3800054280,1
32,3800001711,1
18,1780010508,1
124,1600062740,1
95,3800054280,1
114,1600062740,1
12,3828125030,1
47,3800004270,1


In [15]:

# Step 15: Understand the MOVE outcome

print("Minimum MOVE:", df["MOVE"].min())
print("Maximum MOVE:", df["MOVE"].max())
print("Number of unique MOVE values:", df["MOVE"].nunique())

print("\nNumber of zero MOVE values:", (df["MOVE"] == 0).sum())

print("\nMOVE percentiles:")
display(df["MOVE"].quantile([0.25, 0.50, 0.75, 0.90, 0.99, 0.999]))


Minimum MOVE: 1
Maximum MOVE: 18688
Number of unique MOVE values: 1653

Number of zero MOVE values: 0

MOVE percentiles:


,MOVE
0.250,7.0
0.500,13.0
0.750,21.0
0.900,34.0
0.990,135.0
0.999,569.0


In [16]:

# Step 16: Check price and movement variation

pair_variation = (
    df.groupby(["STORE", "UPC"])
    .agg(
        price_std=("PRICE", "std"),
        move_std=("MOVE", "std"),
        observations=("WEEK", "count")
    )
)

print("Pairs with price variation:")
print((pair_variation["price_std"] > 0).sum())

print("\nPairs with movement variation:")
print((pair_variation["move_std"] > 0).sum())

print("\nPairs with both price and movement variation:")
print(
    (
        (pair_variation["price_std"] > 0) &
        (pair_variation["move_std"] > 0)
    ).sum()
)


Pairs with price variation:
31202

Pairs with movement variation:
35728

Pairs with both price and movement variation:
31180
